# Baseline Models

Before deploying complex forecasting models (ARIMA, VAR, machine learning, etc.),
it is essential to establish **baseline benchmarks**. A baseline is a simple model
that any serious forecasting method should be able to beat.

**Why baselines matter:**
- If your complex model can't beat a simple baseline, something is wrong.
- Baselines provide a lower bound for acceptable performance.
- They help you understand the inherent difficulty of the forecasting problem.

**Models covered:**
1. **Naive Forecast** (Random Walk)
2. **Seasonal Naive**
3. **Drift Method** (Random Walk with Drift)
4. **Simple Moving Average (SMA)**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecastbox.metrics import mae, rmse, mape, mase
from forecastbox.cv import expanding_window_cv
from forecastbox.auto._baselines import NaiveBaseline, SeasonalNaiveBaseline, DriftBaseline

import sys
sys.path.insert(0, "../..")
from utils.helpers import load_macro_brazil, load_macro_us, plot_series, plot_forecast

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
# Load data
df_brazil = load_macro_brazil()
gdp = df_brazil["gdp_growth"].dropna()
inflation = df_brazil["inflation"].dropna()

# Train/test split (80/20)
n = len(gdp)
split = int(0.8 * n)
gdp_train, gdp_test = gdp.iloc[:split], gdp.iloc[split:]
inf_train, inf_test = inflation.iloc[:split], inflation.iloc[split:]
h = len(gdp_test)

print(f"GDP growth: {n} total, {split} train, {n - split} test")
print(f"Inflation:  {len(inflation)} total, {split} train, {len(inflation) - split} test")
print(f"Forecast horizon: {h} months")

## 1. Naive Forecast (Random Walk)

The simplest possible forecast: predict that the next value equals the last observed value.

$$\hat{y}_{T+h} = y_T \quad \text{for all } h$$

This is equivalent to saying "tomorrow will be like today." Despite its simplicity,
the naive forecast is surprisingly hard to beat for many economic and financial series
(this is related to the efficient market hypothesis).

In [ ]:
# Naive forecast for GDP growth
naive = NaiveBaseline()
naive.fit(gdp_train)
naive_fc = naive.forecast(h)

print(f"Last training value: {gdp_train.iloc[-1]:.4f}")
print(f"Naive forecast (constant): {naive_fc.point[0]:.4f}")

# Evaluate
naive_mae = mae(gdp_test.values, naive_fc.point)
naive_rmse = rmse(gdp_test.values, naive_fc.point)
print(f"\nNaive MAE:  {naive_mae:.4f}")
print(f"Naive RMSE: {naive_rmse:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gdp_train.index, gdp_train.values, label="Training", color="#2196F3")
ax.plot(gdp_test.index, gdp_test.values, label="Actual (test)", color="#4CAF50")
ax.plot(gdp_test.index, naive_fc.point, label="Naive forecast", color="#F44336",
        linestyle="--", linewidth=2)
ax.axvline(gdp_test.index[0], color="gray", linestyle=":", alpha=0.7)
ax.set_title("Naive Forecast: GDP Growth", fontsize=14)
ax.set_ylabel("GDP Growth (%)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Seasonal Naive

For data with seasonality, the seasonal naive forecast repeats the **last observed seasonal cycle**:

$$\hat{y}_{T+h} = y_{T+h-m} \quad \text{where } m \text{ is the seasonal period}$$

For monthly data with annual seasonality, $m = 12$: the forecast for January next year
equals the value from January this year.

This baseline is appropriate when the series exhibits clear seasonal patterns.

In [ ]:
# Seasonal naive for inflation (monthly data, period=12)
snaive = SeasonalNaiveBaseline(seasonal_period=12)
snaive.fit(inf_train)
snaive_fc = snaive.forecast(len(inf_test))

snaive_mae = mae(inf_test.values, snaive_fc.point)
snaive_rmse = rmse(inf_test.values, snaive_fc.point)
print(f"Seasonal Naive (m=12) for Inflation:")
print(f"  MAE:  {snaive_mae:.4f}")
print(f"  RMSE: {snaive_rmse:.4f}")

# Compare with simple naive
naive_inf = NaiveBaseline()
naive_inf.fit(inf_train)
naive_inf_fc = naive_inf.forecast(len(inf_test))
naive_inf_mae = mae(inf_test.values, naive_inf_fc.point)
print(f"\nSimple Naive for Inflation:")
print(f"  MAE:  {naive_inf_mae:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(inf_train.index[-36:], inf_train.values[-36:], label="Training (last 3yr)",
        color="#2196F3")
ax.plot(inf_test.index, inf_test.values, label="Actual", color="#4CAF50")
ax.plot(inf_test.index, snaive_fc.point, label="Seasonal Naive", color="#F44336",
        linestyle="--", linewidth=2)
ax.plot(inf_test.index, naive_inf_fc.point, label="Simple Naive", color="#FF9800",
        linestyle=":", linewidth=2)
ax.axvline(inf_test.index[0], color="gray", linestyle=":", alpha=0.7)
ax.set_title("Seasonal Naive vs Simple Naive: Inflation", fontsize=14)
ax.set_ylabel("Inflation (%)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Drift Method

The drift method is a random walk with a **linear trend** (drift):

$$\hat{y}_{T+h} = y_T + h \cdot \left(\frac{y_T - y_1}{T - 1}\right)$$

The drift term $\frac{y_T - y_1}{T - 1}$ is the average change per period.
This model extrapolates a straight line from the first to the last observation.

**Best for:** series with a clear upward or downward trend.

In [ ]:
# Drift forecast for GDP growth
drift = DriftBaseline()
drift.fit(gdp_train)
drift_fc = drift.forecast(h)

print(f"Drift estimate: {drift._drift:.6f} per month")
print(f"Last training value: {gdp_train.iloc[-1]:.4f}")
print(f"Drift forecast range: [{drift_fc.point[0]:.4f}, {drift_fc.point[-1]:.4f}]")

drift_mae = mae(gdp_test.values, drift_fc.point)
drift_rmse = rmse(gdp_test.values, drift_fc.point)
print(f"\nDrift MAE:  {drift_mae:.4f}")
print(f"Drift RMSE: {drift_rmse:.4f}")

# Plot: drift vs naive
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gdp_train.index[-36:], gdp_train.values[-36:], label="Training (last 3yr)",
        color="#2196F3")
ax.plot(gdp_test.index, gdp_test.values, label="Actual", color="#4CAF50")
ax.plot(gdp_test.index, drift_fc.point, label="Drift", color="#9C27B0",
        linestyle="--", linewidth=2)
ax.plot(gdp_test.index, naive_fc.point, label="Naive", color="#F44336",
        linestyle=":", linewidth=2)
ax.axvline(gdp_test.index[0], color="gray", linestyle=":", alpha=0.7)
ax.set_title("Drift vs Naive: GDP Growth", fontsize=14)
ax.set_ylabel("GDP Growth (%)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Simple Moving Average

The Simple Moving Average (SMA) forecasts the next value as the average of the last $k$ observations:

$$\hat{y}_{T+1} = \frac{1}{k}\sum_{i=0}^{k-1} y_{T-i}$$

**Key trade-off:**
- **Small $k$** (e.g., 3): responds quickly to changes, but noisy
- **Large $k$** (e.g., 12): smoother, but slow to adapt

SMA is a natural baseline that captures the recent level of a series without assuming
any trend or seasonality.

In [ ]:
# SMA with different window sizes
windows = [3, 6, 12]
sma_results = {}

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gdp_test.index, gdp_test.values, label="Actual", color="#4CAF50", linewidth=2)

colors = ["#F44336", "#FF9800", "#9C27B0"]
for k, color in zip(windows, colors):
    # SMA forecast: use the mean of last k training values as a constant forecast
    sma_value = gdp_train.iloc[-k:].mean()
    sma_pred = np.full(h, sma_value)

    sma_mae = mae(gdp_test.values, sma_pred)
    sma_rmse = rmse(gdp_test.values, sma_pred)
    sma_results[k] = {"MAE": sma_mae, "RMSE": sma_rmse, "forecast": sma_value}

    ax.plot(gdp_test.index, sma_pred, label=f"SMA-{k} (MAE={sma_mae:.4f})",
            color=color, linestyle="--", linewidth=1.5)
    print(f"SMA-{k:2d}: forecast={sma_value:.4f}, MAE={sma_mae:.4f}, RMSE={sma_rmse:.4f}")

ax.axvline(gdp_test.index[0], color="gray", linestyle=":", alpha=0.7)
ax.set_title("Simple Moving Average: GDP Growth", fontsize=14)
ax.set_ylabel("GDP Growth (%)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Comparing All Baselines

Now let's compare all baseline models using cross-validation to get a more robust
performance estimate. We use expanding window CV with an initial window of 60 months.

In [ ]:
# Define model functions for CV
def naive_fn(train: pd.Series) -> np.ndarray:
    model = NaiveBaseline()
    model.fit(train)
    return model.forecast(12).point

def snaive_fn(train: pd.Series) -> np.ndarray:
    model = SeasonalNaiveBaseline(seasonal_period=12)
    model.fit(train)
    return model.forecast(12).point

def drift_fn(train: pd.Series) -> np.ndarray:
    model = DriftBaseline()
    model.fit(train)
    return model.forecast(12).point

def sma3_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-3:].mean())

def sma6_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-6:].mean())

def sma12_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-12:].mean())

# Run CV for each model
models = {
    "Naive": naive_fn,
    "Seasonal Naive": snaive_fn,
    "Drift": drift_fn,
    "SMA-3": sma3_fn,
    "SMA-6": sma6_fn,
    "SMA-12": sma12_fn,
}

cv_results = {}
for name, fn in models.items():
    result = expanding_window_cv(
        data=gdp,
        model_fn=fn,
        initial_window=60,
        horizon=12,
        step=6,
    )
    cv_results[name] = result

# Build comparison table
rows = []
for name, res in cv_results.items():
    rows.append({
        "Model": name,
        "CV MAE": res.metrics_overall.get("mae", np.nan),
        "CV RMSE": res.metrics_overall.get("rmse", np.nan),
        "Folds": res.n_folds,
    })

comparison = pd.DataFrame(rows).set_index("Model").sort_values("CV MAE")
print("Baseline Comparison: GDP Growth (Expanding Window CV)")
print("=" * 55)
print(comparison.round(4))
print(f"\nBest model by MAE: {comparison.index[0]}")

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(10, 6))

model_names = comparison.index.tolist()
mae_values = comparison["CV MAE"].values
rmse_values = comparison["CV RMSE"].values

x = np.arange(len(model_names))
width = 0.35

ax.barh(x - width/2, mae_values, width, label="MAE", color="#2196F3", alpha=0.8)
ax.barh(x + width/2, rmse_values, width, label="RMSE", color="#F44336", alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(model_names)
ax.set_xlabel("Error")
ax.set_title("Baseline Model Comparison: GDP Growth", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

## Exercise 1: Find the best baseline for unemployment

Compare all baseline models (naive, seasonal naive, drift, SMA-3, SMA-6, SMA-12)
for Brazil's unemployment series using expanding window cross-validation.
Which model performs best? Does it make sense given the characteristics of unemployment data?

In [ ]:
# SOLUTION: Exercise 1 - Compare baselines for BR unemployment

# Step 1: Load unemployment data
unemployment = df_brazil["unemployment"].dropna()
print(f"Unemployment series: {len(unemployment)} observations")
print(f"Period: {unemployment.index[0].strftime('%Y-%m')} to {unemployment.index[-1].strftime('%Y-%m')}")
print(f"Mean: {unemployment.mean():.4f}, Std: {unemployment.std():.4f}")

# Step 2: Train/test split (80/20)
n_u = len(unemployment)
split_u = int(0.8 * n_u)
unemp_train = unemployment.iloc[:split_u]
unemp_test = unemployment.iloc[split_u:]
h_u = len(unemp_test)
print(f"Train: {split_u}, Test: {h_u}")

# Step 3: Define all baseline model functions for CV
def naive_fn(train: pd.Series) -> np.ndarray:
    model = NaiveBaseline()
    model.fit(train)
    return model.forecast(12).point

def snaive_fn(train: pd.Series) -> np.ndarray:
    model = SeasonalNaiveBaseline(seasonal_period=12)
    model.fit(train)
    return model.forecast(12).point

def drift_fn(train: pd.Series) -> np.ndarray:
    model = DriftBaseline()
    model.fit(train)
    return model.forecast(12).point

def sma3_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-3:].mean())

def sma6_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-6:].mean())

def sma12_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-12:].mean())

# Step 4: Run expanding window CV for each model
models = {
    "Naive": naive_fn,
    "Seasonal Naive (m=12)": snaive_fn,
    "Drift": drift_fn,
    "SMA-3": sma3_fn,
    "SMA-6": sma6_fn,
    "SMA-12": sma12_fn,
}

cv_results_u = {}
for name, fn in models.items():
    result = expanding_window_cv(
        data=unemployment,
        model_fn=fn,
        initial_window=60,
        horizon=12,
        step=6,
    )
    cv_results_u[name] = result

# Step 5: Build comparison table with MAE, RMSE, and MASE
# Expected values: Naive MAE~0.3016, Drift MAE~0.3154, SMA-3 MAE~0.3177, SNaive MAE~0.4411
rows = []
for name, res in cv_results_u.items():
    # Also compute MASE on a simple train/test split
    if "Naive" == name:
        m = NaiveBaseline(); m.fit(unemp_train); fc = m.forecast(h_u).point
    elif "Seasonal" in name:
        m = SeasonalNaiveBaseline(seasonal_period=12); m.fit(unemp_train); fc = m.forecast(h_u).point
    elif "Drift" == name:
        m = DriftBaseline(); m.fit(unemp_train); fc = m.forecast(h_u).point
    elif "SMA-3" == name:
        fc = np.full(h_u, unemp_train.iloc[-3:].mean())
    elif "SMA-6" == name:
        fc = np.full(h_u, unemp_train.iloc[-6:].mean())
    else:
        fc = np.full(h_u, unemp_train.iloc[-12:].mean())

    mase_val = mase(unemp_test.values, fc, training_series=unemp_train.values)
    rows.append({
        "Model": name,
        "CV MAE": res.metrics_overall.get("mae", np.nan),
        "CV RMSE": res.metrics_overall.get("rmse", np.nan),
        "MASE": mase_val,
        "Folds": res.n_folds,
    })

comparison_u = pd.DataFrame(rows).set_index("Model").sort_values("CV MAE")
print("\nBaseline Comparison: BR Unemployment (Expanding Window CV)")
print("=" * 65)
print(comparison_u.round(4))
print(f"\nBest model by MAE: {comparison_u.index[0]}")

# Step 6: Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of MAE and RMSE
model_names = comparison_u.index.tolist()
x = np.arange(len(model_names))
width = 0.35

axes[0].barh(x - width/2, comparison_u["CV MAE"].values, width, label="MAE", color="#2196F3", alpha=0.8)
axes[0].barh(x + width/2, comparison_u["CV RMSE"].values, width, label="RMSE", color="#F44336", alpha=0.8)
axes[0].set_yticks(x)
axes[0].set_yticklabels(model_names)
axes[0].set_xlabel("Error")
axes[0].set_title("Baseline Comparison: Unemployment", fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="x")

# Plot actual vs best model forecast
best_model = NaiveBaseline()
best_model.fit(unemp_train)
best_fc = best_model.forecast(h_u)
axes[1].plot(unemp_train.index[-36:], unemp_train.values[-36:], label="Training (last 3yr)", color="#2196F3")
axes[1].plot(unemp_test.index, unemp_test.values, label="Actual", color="#4CAF50", linewidth=2)
axes[1].plot(unemp_test.index, best_fc.point, label="Naive (best)", color="#F44336", linestyle="--", linewidth=2)
axes[1].axvline(unemp_test.index[0], color="gray", linestyle=":", alpha=0.7)
axes[1].set_title("Best Baseline: Naive for Unemployment", fontsize=13)
axes[1].set_ylabel("Unemployment (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Step 7: Interpretation
print("""
--- Interpretation ---
The Naive forecast performs best for Brazilian unemployment by CV MAE (~0.3016).

This makes economic sense because unemployment is a highly persistent (autocorrelated)
series: unemployment this month is very close to unemployment last month. The series
changes slowly and smoothly, making the "last value" prediction hard to beat.

The Seasonal Naive performs worst because unemployment in this synthetic dataset
does not exhibit strong annual seasonality, so repeating last year's pattern
introduces unnecessary errors.

SMA models smooth out recent values, which slightly hurts performance compared
to naive for this highly persistent series — the last single observation is
more informative than the average of the last k observations.

All MASE values are > 1 because the out-of-sample period is harder to predict
than the in-sample period (which is used for MASE scaling).
""")

## Exercise 2: Build a baseline for US CPI inflation

Load the US dataset (`macro_us.csv`) and build baseline forecasts for `cpi_inflation`.
Compare at least 3 baseline models and visualize the results.

In [ ]:
# SOLUTION: Exercise 2 - Baselines for US CPI inflation with CV temporal

# Step 1: Load US data
df_us = load_macro_us()
cpi = df_us["cpi_inflation"].dropna()
print(f"US CPI Inflation: {len(cpi)} observations")
print(f"Period: {cpi.index[0].strftime('%Y-%m')} to {cpi.index[-1].strftime('%Y-%m')}")
print(f"Mean: {cpi.mean():.4f}, Std: {cpi.std():.4f}")

# Step 2: Train/test split for point estimates
n_cpi = len(cpi)
split_cpi = int(0.8 * n_cpi)
cpi_train = cpi.iloc[:split_cpi]
cpi_test = cpi.iloc[split_cpi:]
h_cpi = len(cpi_test)
print(f"Train: {split_cpi}, Test: {h_cpi}")

# Step 3: Define model functions (reuse from above)
def naive_fn(train: pd.Series) -> np.ndarray:
    model = NaiveBaseline()
    model.fit(train)
    return model.forecast(12).point

def snaive_fn(train: pd.Series) -> np.ndarray:
    model = SeasonalNaiveBaseline(seasonal_period=12)
    model.fit(train)
    return model.forecast(12).point

def drift_fn(train: pd.Series) -> np.ndarray:
    model = DriftBaseline()
    model.fit(train)
    return model.forecast(12).point

def sma3_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-3:].mean())

def sma6_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-6:].mean())

def sma12_fn(train: pd.Series) -> np.ndarray:
    return np.full(12, train.iloc[-12:].mean())

# Step 4: Run expanding window CV for all models
# Expected: SMA-3 MAE~0.1240 (best), Naive MAE~0.1326, SNaive MAE~0.1627
models_cpi = {
    "Naive": naive_fn,
    "Seasonal Naive (m=12)": snaive_fn,
    "Drift": drift_fn,
    "SMA-3": sma3_fn,
    "SMA-6": sma6_fn,
    "SMA-12": sma12_fn,
}

cv_results_cpi = {}
for name, fn in models_cpi.items():
    result = expanding_window_cv(
        data=cpi,
        model_fn=fn,
        initial_window=60,
        horizon=12,
        step=6,
    )
    cv_results_cpi[name] = result

# Step 5: Build comparison table
rows = []
for name, res in cv_results_cpi.items():
    rows.append({
        "Model": name,
        "CV MAE": res.metrics_overall.get("mae", np.nan),
        "CV RMSE": res.metrics_overall.get("rmse", np.nan),
        "Folds": res.n_folds,
    })

comparison_cpi = pd.DataFrame(rows).set_index("Model").sort_values("CV MAE")
print("\nBaseline Comparison: US CPI Inflation (Expanding Window CV)")
print("=" * 65)
print(comparison_cpi.round(4))
print(f"\nBest model by MAE: {comparison_cpi.index[0]}")

# Step 6: Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparison
model_names = comparison_cpi.index.tolist()
x = np.arange(len(model_names))
width = 0.35

axes[0].barh(x - width/2, comparison_cpi["CV MAE"].values, width, label="MAE", color="#2196F3", alpha=0.8)
axes[0].barh(x + width/2, comparison_cpi["CV RMSE"].values, width, label="RMSE", color="#F44336", alpha=0.8)
axes[0].set_yticks(x)
axes[0].set_yticklabels(model_names)
axes[0].set_xlabel("Error")
axes[0].set_title("Baseline Comparison: US CPI Inflation", fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="x")

# Plot actual vs top 3 forecasts
# Generate point forecasts for visualization
forecasts_cpi = {
    "Naive": np.full(h_cpi, cpi_train.iloc[-1]),
    "SMA-3": np.full(h_cpi, cpi_train.iloc[-3:].mean()),
    "Drift": DriftBaseline().fit(cpi_train).forecast(h_cpi).point,
}

colors = {"Naive": "#F44336", "SMA-3": "#FF9800", "Drift": "#9C27B0"}
axes[1].plot(cpi_train.index[-36:], cpi_train.values[-36:], label="Training (last 3yr)", color="#2196F3")
axes[1].plot(cpi_test.index, cpi_test.values, label="Actual", color="#4CAF50", linewidth=2)
for name, fc_pred in forecasts_cpi.items():
    axes[1].plot(cpi_test.index, fc_pred, label=name, color=colors[name],
                 linestyle="--", linewidth=1.5)
axes[1].axvline(cpi_test.index[0], color="gray", linestyle=":", alpha=0.7)
axes[1].set_title("Top Baselines: US CPI Inflation", fontsize=13)
axes[1].set_ylabel("CPI Inflation (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Step 7: Interpretation
print("""
--- Interpretation ---
For US CPI Inflation, SMA-3 achieves the best CV MAE (~0.1240), followed closely
by SMA-6 (~0.1296), SMA-12 (~0.1306), and Naive (~0.1326).

Key observations:
1. SMA models outperform the simple Naive baseline, suggesting that averaging
   recent values provides a better estimate of the inflation level than using
   just the last observation. This is because CPI inflation is mean-reverting.

2. Seasonal Naive performs worst (~0.1627), indicating that annual seasonality
   is not a dominant feature in this inflation series.

3. Drift performs slightly worse than Naive, suggesting that extrapolating a
   linear trend is not helpful for inflation (which fluctuates around a level).

4. The differences between SMA-3, SMA-6, SMA-12, and Naive are small (~0.01),
   indicating that all simple baselines capture the key dynamics reasonably well.
   Beating these baselines with a complex model requires genuine additional
   forecasting power.

5. The small spread in MAE across models confirms that CPI inflation is
   relatively easy to forecast at short horizons with simple methods.
""")